# Feature set

Runnable input for **gradient-boosted trees** predicting `OD_PAX`.

The cutoff is the start of the departure month. `build_features` returns `X` and `y`. `X` does not contain `OD_PAX`, `OD_REV_USD`, `Unnamed: 0`, or `SF_LOAD_TMS`.

Place, route, and month columns are pandas `category` dtype. Deliverable 4 can pass `X` to XGBoost with `enable_categorical=True`, to LightGBM, or to `HistGradientBoostingRegressor` without another encoding step.

This is the frame Deliverable 4 should tune. It does not fit a model.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "raw").exists())
pax = pd.read_csv(root / "data" / "raw" / "AA_PAX_FORECASTING.csv")
airports = pd.read_csv(root / "data" / "raw" / "AA_AIRPORT_INFO.csv")


def decimal_degrees(degrees, minutes, seconds, hemisphere):
    magnitude = degrees.astype(float) + minutes.astype(float) / 60 + seconds.astype(float) / 3600
    sign = hemisphere.astype(str).str.upper().map({"N": 1, "S": -1, "E": 1, "W": -1})
    return magnitude * sign


def airport_lookup(airports):
    """One row per airport. Names and raw clock-position fields are not features."""
    # .to_numpy() keeps each column in row order. Passing the airport code as
    # the DataFrame index would realign those columns onto the codes and blank them.
    lookup = pd.DataFrame(
        {
            "CITY_METRO_IATA_CD": airports["CITY_METRO_IATA_CD"].fillna("MISSING").to_numpy(),
            "STATE_PROVNC_CD": airports["STATE_PROVNC_CD"].fillna("MISSING").to_numpy(),
            "CNTRY_CD": airports["CNTRY_CD"].fillna("MISSING").to_numpy(),
            "WRLD_AREA_DOT_CD": airports["WRLD_AREA_DOT_CD"].astype("int64").astype(str).to_numpy(),
            "LAT": decimal_degrees(
                airports["LAT_DEGREE_QTY"],
                airports["LAT_MINUTE_QTY"],
                airports["LAT_SECOND_QTY"],
                airports["LAT_HEMSPHR_CD"],
            ).to_numpy(),
            "LNG": decimal_degrees(
                airports["LNGTD_DEGREE_QTY"],
                airports["LNGTD_MINUTE_QTY"],
                airports["LNGTD_SECOND_QTY"],
                airports["LNGTD_HEMSPHR_CD"],
            ).to_numpy(),
            # A runway length of 0 is not a real runway. None of those airports
            # appear on the forecasting routes; this keeps the lookup honest.
            "RUNWAY_FT": airports["LNGST_RUNWAY_FT_QTY"].replace(0, np.nan).to_numpy(),
            "ELEVATION_FT": airports["ELEVATN_FT_QTY"].to_numpy(),
        }
    )
    lookup.index = airports["AIRPRT_CD"].to_numpy()
    return lookup[~lookup.index.duplicated()]


def build_features(pax, airports):
    lookup = airport_lookup(airports)
    frame = pax.drop(columns=["Unnamed: 0", "SF_LOAD_TMS", "OD_REV_USD"]).copy()
    y = frame.pop("OD_PAX")
    month = frame["DEP_MONTH"].astype(int)
    features = frame.join(lookup.add_prefix("ORG_"), on="ORG_AIRPRT")
    features = features.join(lookup.add_prefix("DES_"), on="DES_AIRPRT")
    features["DEP_MONTH"] = month.astype(str)
    features["MONTH_SIN"] = np.sin(2 * np.pi * month / 12)
    features["MONTH_COS"] = np.cos(2 * np.pi * month / 12)
    categorical = [
        "DEP_MONTH",
        "ORG_AIRPRT",
        "DES_AIRPRT",
        "ORG_CITY_METRO_IATA_CD",
        "DES_CITY_METRO_IATA_CD",
        "ORG_STATE_PROVNC_CD",
        "DES_STATE_PROVNC_CD",
        "ORG_CNTRY_CD",
        "DES_CNTRY_CD",
        "ORG_WRLD_AREA_DOT_CD",
        "DES_WRLD_AREA_DOT_CD",
    ]
    for column in categorical:
        features[column] = features[column].astype("category")
    return features, y


X, y = build_features(pax, airports)
assert len(X) == len(pax) == len(y)
assert "OD_PAX" not in X.columns
assert "OD_REV_USD" not in X.columns
assert "Unnamed: 0" not in X.columns
assert "SF_LOAD_TMS" not in X.columns
assert X["ORG_CNTRY_CD"].notna().all()
assert X["DES_CNTRY_CD"].notna().all()
assert X.select_dtypes("category").shape[1] == 11
print(f"X: {X.shape[0]:,} rows, {X.shape[1]} columns")
print("Columns:", list(X.columns))
print("Category columns:", list(X.select_dtypes("category").columns))
print("Missing cells in X:", int(X.isna().sum().sum()))


The build keeps every forecasting row (1,533,522). Both joins match. `X` has 22 columns and no missing cells on those rows: 11 categories and 11 numbers. Four airports in the lookup have a runway length of 0, and none of them is an origin or destination in the forecasting file.

## Why this model

Gradient-boosted trees. The target is skewed (skew 5.32, mean about 2,765 against a median of about 503). The useful inputs are airport categories plus a few fixed numeric attributes. Same-month revenue correlates with passengers at about 0.76 and is excluded, so the model is not being chosen because it can memorize that column.

## Kept

| Feature | Why it stays |
|---|---|
| `DEP_YEAR` | Known at the cutoff. Carries the 2019-2026 level, including the thinner 2020. |
| `DEP_MONTH` | Known at the cutoff. Stored as a category because June and July are higher and the row-level relationship is not a straight line. |
| `MONTH_SIN`, `MONTH_COS` | The same month, encoded so December sits next to January. |
| `ORG_AIRPRT`, `DES_AIRPRT` | The route. 150 origins and 150 destinations. This is the main split a tree will use. |
| `ORG_` / `DES_` `CITY_METRO_IATA_CD` | Metro area, coarser than the airport code. |
| `ORG_` / `DES_` `STATE_PROVNC_CD` | Kept, with blanks filled as `MISSING`. Every US airport has a state. The blanks are non-US airports, which is known before departure. Filling them with a guess would invent states. |
| `ORG_` / `DES_` `CNTRY_CD` | Country of the airport. Eight lookup rows are blank and become `MISSING`. |
| `ORG_` / `DES_` `WRLD_AREA_DOT_CD` | DOT world-area code. A region bucket above country. |
| `ORG_` / `DES_` `LAT`, `LNG` | Decimal degrees. Degree, minute, second, and hemisphere are the same location split into four fields, so those raw parts are not also kept. |
| `ORG_` / `DES_` `RUNWAY_FT` | Longest runway. A stand-in for aircraft the airport can take. |
| `ORG_` / `DES_` `ELEVATION_FT` | Field elevation. Fixed, and negative values are real (below sea level). |

## Dropped

| Column | Why it goes |
|---|---|
| `OD_REV_USD` | Same-month revenue from the passengers being predicted. Not known at the cutoff. |
| `Unnamed: 0` | Export row number, `0` through `n-1` in file order. |
| `SF_LOAD_TMS` | One Snowflake load timestamp on every row of each file. |
| `AIRPRT_NM` | The name of the airport code. One string per code, and the code is already kept. |
| `AIRPRT_CD` as its own column | Used only to attach origin and destination attributes. |
| Raw latitude and longitude parts | Replaced by `LAT` and `LNG`. |
| `OD_PAX` inside `X` | It is `y`. |

`build_features(pax, airports)` is the handoff to Deliverable 4.